In [1]:
# %% [markdown]
# # Shapes de la cuenca asignada a cada pozo (trended y detrended)
# Entrada: `gw_memory_wide_best_basin.csv`. Salida: un archivo por variante, una fila (polígono) por pozo.
# - Por defecto el polígono es la **cuenca contribuyente completa** (cuenca del pozo + todas las de aguas arriba, disueltas),
#   que es el área sobre la que se calculó la precipitación. Con `incluir_aguas_arriba = False` sale solo la cuenca del pozo.

# %%
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import box
from pathlib import Path

# === Parámetros ===
csv_in = Path('csv_memory/gw_memory_wide_best_basin.csv')
root_gis = r"C:/DOCUMENTS/EARTH_SCIENCES/CLIMA/2027_Alvarez-Garreton/R_nue1/GIS"
out_dir = Path('shapes_cuencas')
out_dir.mkdir(parents=True, exist_ok=True)
incluir_aguas_arriba = True
formatos = ['gpkg']             # gpkg: nombres completos. shp: nombres cortos (máx. 10 caracteres)
bbox = (-74, -43, -66, -17)            # lon_min, lat_min, lon_max, lat_max (solo acelera la lectura)
tol_area = 0.01                        # tolerancia relativa del área del polígono vs UP_AREA

variantes = {'raw': 'trended', 'dt': 'detrended'}

# %%
df = pd.read_csv(csv_in, dtype={'well_id': str, 'HYBAS_ID_upstream_raw': str, 'HYBAS_ID_upstream_dt': str})



def parse_ids(txt):
    """'6050029280;6050029281.0' -> [6050029280, 6050029281]  (tolera IDs escritos con '.0')"""
    if pd.isna(txt):
        return []
    return [int(round(float(i))) for i in str(txt).split(';') if i.strip() != '']


# Limpiar las listas de cuencas (quita los '.0') para que también salgan limpias en los archivos
for v in variantes:
    col = f'HYBAS_ID_upstream_{v}'
    df[col] = df[col].map(lambda t: ';'.join(str(i) for i in parse_ids(t)) if pd.notna(t) else t)

_cache = {}


def read_basins(lvl):
    if lvl in _cache:
        return _cache[lvl]
    shp = os.path.join(root_gis, f"hybas_sa_lev{lvl:02d}_v1c.shp")
    try:
        b = gpd.read_file(shp, bbox=bbox)
    except Exception as e:                       # índice .sbn dañado: leer todo y filtrar
        print(f"   bbox falló en nivel {lvl} ({str(e)[:40]}...), leyendo completo")
        b = gpd.read_file(shp)
        b = b[b.intersects(box(*bbox))]
    b = b.to_crs(4326) if (b.crs is not None and b.crs.to_epsg() != 4326) else b
    b['HYBAS_ID'] = b['HYBAS_ID'].astype('int64')
    _cache[lvl] = b.drop_duplicates('HYBAS_ID').set_index('HYBAS_ID')
    return _cache[lvl]


# %%
for v, nombre in variantes.items():
    d = df.dropna(subset=[f'nivel_{v}']).copy()
    geoms, faltan = [], []

    for _, r in d.iterrows():
        b = read_basins(int(r[f'nivel_{v}']))
        if incluir_aguas_arriba:
            ids = parse_ids(r[f'HYBAS_ID_upstream_{v}'])
        else:
            ids = [int(round(float(r[f'HYBAS_ID_{v}'])))]
        ok = [i for i in ids if i in b.index]
        if len(ok) < len(ids):
            faltan.append((r['well_id'], len(ids) - len(ok)))
        geoms.append(shapely.union_all(b.loc[ok].geometry.values) if ok else None)

    gdf = gpd.GeoDataFrame(
        {
            'well_id': d['well_id'].values,
            'nivel': d[f'nivel_{v}'].astype(int).values,
            'HYBAS_ID': d[f'HYBAS_ID_{v}'].astype('float64').round().astype('int64').values,
            'UP_AREA': d[f'UP_AREA_{v}'].values,
            'n_aguasarr': d[f'n_cuencas_aguas_arriba_{v}'].values,
            'cobert_P': d[f'cobertura_P_{v}'].values,
            'r2': d[f'r2_{v}'].values,
            'memory_iso': d[f'memory_iso_{v}'].values,
        },
        geometry=geoms, crs=4326)
    gdf = gdf[~gdf.geometry.isna()].copy()

    # Control: área del polígono (proyección de igual área) vs UP_AREA de HydroBASINS
    if incluir_aguas_arriba:
        a = gdf.to_crs(6933).area / 1e6
        gdf['area_km2'] = a.values
        dif = (gdf['area_km2'] - gdf['UP_AREA']).abs() / gdf['UP_AREA']
        print(f"[{nombre}] {len(gdf)} polígonos; dif. máx. de área vs UP_AREA = {dif.max():.2%}; "
              f"{(dif > tol_area).sum()} fuera de tolerancia")
    if faltan:
        print(f"[{nombre}] pozos con cuencas no encontradas en el shapefile: {faltan[:10]}")

    for fmt in formatos:
        base = out_dir / f"cuencas_pozos_{nombre}"
        if fmt == 'gpkg':
            full = gdf.rename(columns={'n_aguasarr': 'n_cuencas_aguas_arriba', 'cobert_P': 'cobertura_P'})
            full['HYBAS_ID_upstream'] = d.set_index('well_id').loc[full['well_id'], f'HYBAS_ID_upstream_{v}'].values
            full.to_file(f"{base}.gpkg", driver='GPKG')
        elif fmt == 'shp':
            gdf.to_file(f"{base}.shp")          # nombres de campo <= 10 caracteres
    print(f"[{nombre}] guardado en {out_dir}")

print("\nLR models Finished! \nFinito! \nTerminado!\n")

[trended] 503 polígonos; dif. máx. de área vs UP_AREA = 0.40%; 0 fuera de tolerancia
[trended] guardado en shapes_cuencas
[detrended] 503 polígonos; dif. máx. de área vs UP_AREA = 0.39%; 0 fuera de tolerancia
[detrended] guardado en shapes_cuencas

LR models Finished! 
Finito! 
Terminado!

